# Módulo 5: Estadística Descriptiva, Agregación y Escalamiento de Datos

## 🎯 Objetivos de Aprendizaje
Al finalizar este cuaderno, serás capaz de:
1. Calcular operaciones de **reducción** (`np.sum`, `np.prod`) y **acumulación** (`np.cumsum`, `np.cumprod`) sobre vectores y matrices.
2. Obtener métricas de extremos (`min`, `max`, `argmin`, `argmax`, rango `ptp`), tendencia central (`mean`, `median`) y dispersión/posición (`std`, `var`, `percentile`) a nivel **global**, **por columnas (`axis=0`)** y **por filas (`axis=1`)**.
3. Transformar vectores y matrices a una nueva escala numérica (por ejemplo, normalización proporcional o Min-Max sobre **100**) aplicando *Broadcasting* y preservando la dimensionalidad mediante `keepdims=True`.
4. Evitar errores estadísticos comunes, como confundir desviación estándar poblacional (`ddof=0`) con muestral (`ddof=1`) o distorsionar escalas por ejes incorrectos.

---

## 🗺️ Mapa Mental del Módulo

```text
               [ Reducción, Estadística y Escalas en NumPy ]
                                     │
       ┌─────────────────────────────┼─────────────────────────────┐
       ▼                             ▼                             ▼
1. AGREGACIÓN Y ACUMULACIÓN   2. ESTADÍSTICA POR EJES       3. ESCALAMIENTO (NORMALIZACIÓN)
├── Reducción: sum, prod      ├── axis=None (Global)        ├── Escala sobre 100 (Máximo)
├── Acumulación: cumsum       ├── axis=0 (Colapso vertical) ├── Escala Min-Max [0, 100]
└── Acumulación: cumprod      └── axis=1 (Colapso horizontal) └── keepdims=True (Broadcasting)

In [1]:
# Importar numpy
import numpy as np

## 5.1 Reducción y Acumulación: `np.sum`, `np.prod`, `np.cumsum` y `np.cumprod`

En NumPy, las funciones de agregación operan sobre un eje y lo **colapsan**:
* **Sin especificar eje (`axis=None`):** Opera sobre todos los elementos del arreglo y devuelve un único escalar.
* **`axis=0` (Colapso vertical):** Recorre las filas hacia abajo, devolviendo **un resultado por cada columna**.
* **`axis=1` (Colapso horizontal):** Recorre las columnas hacia la derecha, devolviendo **un resultado por cada fila**.

### Diferencia entre Reducción y Acumulación
* **Reducción (`np.sum`, `np.prod`):** Comprime un eje de longitud $N$ en un único valor (suma total o producto total). Reduce el rango (`ndim`) en 1.
* **Acumulación (`np.cumsum`, `np.cumprod`):** Mantiene exactamente el mismo `shape` original, registrando el subtotal acumulado paso a paso a lo largo del eje indicado.

### Sintaxis Mínima

In [5]:
# Vector 1D: Flujo de caja neto mensual ($) durante 10 meses
flujo_mensual = np.array([1200, -300, 850, 1500, -450, 900, 1100, -200, 1400, 1600])

print("Flujo mensual        :", flujo_mensual)
print("Suma total anual     :", np.sum(flujo_mensual))
print("Saldo acumulado (1D) :", np.cumsum(flujo_mensual))

# Vector 1D: Factores de crecimiento anual (ej. 1.05 = +5%, 0.98 = -2%)
tasas_crecimiento = np.array([1.05, 1.08, 0.98, 1.10, 1.04])
print("\nFactor acumulado compuesto (cumprod):", np.round(np.cumprod(tasas_crecimiento), 4))
print("Factor total al final del periodo (prod):", np.round(np.prod(tasas_crecimiento), 4))

# Matriz 2D (4x5): Producción diaria (unidades) de 4 líneas (filas) en 5 días (columnas)
produccion = np.array([
    [100, 110, 105, 120, 115],  # Línea 0
    [80,  85,  90,  88,  92],   # Línea 1
    [150, 145, 160, 155, 170],  # Línea 2
    [95,  100, 98,  105, 110]   # Línea 3
])
print("\nMatriz global total (4x5):\n", produccion)

print("\nProducción global total (axis=None):", np.sum(produccion))
print("Producción total por DÍA (axis=0, 5 columnas):", np.sum(produccion, axis=0))
print("Producción total por LÍNEA (axis=1, 4 filas):", np.sum(produccion, axis=1))

# Producción acumulada día a día por cada línea (mantiene shape 4x5)
acumulado_semanal = np.cumsum(produccion, axis=1)
print("\nProducción acumulada día a día por línea (axis=1):\n", acumulado_semanal)

Flujo mensual        : [1200 -300  850 1500 -450  900 1100 -200 1400 1600]
Suma total anual     : 7600
Saldo acumulado (1D) : [1200  900 1750 3250 2800 3700 4800 4600 6000 7600]

Factor acumulado compuesto (cumprod): [1.05   1.134  1.1113 1.2225 1.2714]
Factor total al final del periodo (prod): 1.2714

Matriz global total (4x5):
 [[100 110 105 120 115]
 [ 80  85  90  88  92]
 [150 145 160 155 170]
 [ 95 100  98 105 110]]

Producción global total (axis=None): 2273
Producción total por DÍA (axis=0, 5 columnas): [425 440 453 468 487]
Producción total por LÍNEA (axis=1, 4 filas): [550 435 780 508]

Producción acumulada día a día por línea (axis=1):
 [[100 210 315 435 550]
 [ 80 165 255 343 435]
 [150 295 455 610 780]
 [ 95 195 293 398 508]]


:::{note} Regla Nemotécnica para `axis=0` vs. `axis=1`
El parámetro `axis` indica **qué dimensión desaparece (se comprime)** al aplicar la operación:
* Si una matriz tiene forma `(4, 5)` y aplicas `np.sum(M, axis=0)`, desaparece el `4` (filas) y obtienes un vector de forma `(5,)` (un valor por columna).
* Si aplicas `np.sum(M, axis=1)`, desaparece el `5` (columnas) y obtienes un vector de forma `(4,)` (un valor por fila).
:::

:::{warning} Trampa Común: Desbordamiento entero (`Overflow`) en `np.prod` y `np.cumprod`
El producto acumulado crece exponencialmente. Si trabajas con matrices de enteros pequeños (`int32`), `np.prod` puede superar el límite máximo ($2.14 \times 10^9$) y devolver números negativos erróneos sin lanzar excepción. Si esperas productos grandes, fuerza `dtype=np.float64` o `dtype=np.int64`.
:::

### 🧩 Micro-reto 5.1
Dada la matriz `gastos` de tamaño $3 \times 4$ (3 departamentos $\times$ 4 trimestres) en la celda inferior:
1. Calcula el **gasto total por trimestre** (debes obtener 4 valores).
2. Calcula el **gasto total anual por departamento** (debes obtener 3 valores).
3. Genera una matriz de $3 \times 4$ llamada `gasto_acumulado_dept` que muestre cómo se fue acumulando el gasto de cada departamento trimestre tras trimestre.

In [6]:
gastos = np.array([
    [4500, 4800, 5100, 5600],
    [3200, 3100, 3400, 3800],
    [6000, 6200, 5900, 6500]
])

# Escribe tu solución aquí:

## 5.2 Estadística Descriptiva: Extremos, Tendencia Central, Dispersión y Posición

NumPy implementa todas las métricas descriptivas fundamentales con soporte nativo para cálculo global (`axis=None`), por columnas (`axis=0`) o por filas (`axis=1`):

* **Extremos y Rango:**
  * `np.min(arr, axis=...)` y `np.max(arr, axis=...)`: Valores mínimo y máximo.
  * `np.argmin(arr, axis=...)` y `np.argmax(arr, axis=...)`: **Índices** (posiciones) donde se ubican el mínimo y el máximo.
  * `np.ptp(arr, axis=...)`: Rango dinámico (*Peak-to-Peak*), definido exactamente como $\max - \min$.
* **Tendencia Central:**
  * `np.mean(arr, axis=...)`: Media aritmética (promedio).
  * `np.median(arr, axis=...)`: Mediana (percentil 50; robusta frente a valores atípicos).
* **Dispersión y Cuantiles:**
  * `np.var(arr, axis=...)`: Varianza.
  * `np.std(arr, axis=...)`: Desviación estándar ($\sqrt{\text{var}}$).
  * `np.percentile(arr, q, axis=...)`: Percentil `q` (donde `q` está entre `0` y `100`, ej. `25`, `50`, `75`, `90`).

### Sintaxis Mínima

In [12]:
# Matriz 4x5: Tiempos de respuesta (ms) de 4 servidores (filas) en 5 pruebas de carga (columnas)
tiempos_ms = np.array([
    [120, 125, 118, 130, 122],  # Servidor 0 (muy estable)
    [110, 115, 112, 450, 118],  # Servidor 1 (con un pico atípico de 450 ms)
    [140, 142, 138, 145, 141],  # Servidor 2 (estable)
    [95,  102, 98,  105, 100]   # Servidor 3 (el más rápido)
])
print("Matriz tiempos (4x5):\n", tiempos_ms)

# 1. Extremos globales y por ejes
print(f"\nTiempo mínimo global: {np.min(tiempos_ms)} ms | Máximo global: {np.max(tiempos_ms)} ms")
print("Mínimo por prueba (axis=0):", np.min(tiempos_ms, axis=0))
print("Máximo por servidor (axis=1):", np.max(tiempos_ms, axis=1))
print("Rango dinámico (max - min) por servidor (axis=1):", np.ptp(tiempos_ms, axis=1))

# 3. Tendencia central: Media vs. Mediana (observa el impacto del outlier 450 en Servidor 1)
medias_servidor = np.mean(tiempos_ms, axis=1)
medianas_servidor = np.median(tiempos_ms, axis=1)
print("\nMedia por servidor   :", np.round(medias_servidor, 1))
print("Mediana por servidor :", medianas_servidor)

# 4. Dispersión (Desviación estándar y Varianza) y Percentiles
std_servidor = np.std(tiempos_ms, axis=1)
var_servidor = np.var(tiempos_ms, axis=1)
p90_global = np.percentile(tiempos_ms, 90)
cuartiles_global = np.percentile(tiempos_ms, [25, 50, 75])

print("\nDesviación estándar por servidor:", np.round(std_servidor, 2))
print("Varianza por servidor           :", np.round(var_servidor, 2))
print(f"Percentil 90 global: {p90_global:.1f} ms | Cuartiles (Q1, Q2, Q3): {cuartiles_global}")

Matriz tiempos (4x5):
 [[120 125 118 130 122]
 [110 115 112 450 118]
 [140 142 138 145 141]
 [ 95 102  98 105 100]]

Tiempo mínimo global: 95 ms | Máximo global: 450 ms
Mínimo por prueba (axis=0): [ 95 102  98 105 100]
Máximo por servidor (axis=1): [130 450 145 105]
Rango dinámico (max - min) por servidor (axis=1): [ 12 340   7  10]

Media por servidor   : [123.  181.  141.2 100. ]
Mediana por servidor : [122. 115. 141. 100.]

Desviación estándar por servidor: [  4.2  134.53   2.32   3.41]
Varianza por servidor           : [1.76000e+01 1.80976e+04 5.36000e+00 1.16000e+01]
Percentil 90 global: 142.3 ms | Cuartiles (Q1, Q2, Q3): [108.75 119.   138.5 ]


:::{tip} Manejo de Valores Faltantes (`NaN`)
Si tu matriz contiene al menos un valor `np.nan`, funciones como `np.mean()`, `np.min()` o `np.std()` devolverán `nan`. Para ignorar los valores faltantes en el cálculo, utiliza las variantes seguras: `np.nanmean()`, `np.nanstd()`, `np.nanmin()`, `np.nanmax()` y `np.nanpercentile()`.
:::

### 🧩 Micro-reto 5.2
Dada la matriz `notas_curso` de $5 \times 4$ (5 estudiantes $\times$ 4 exámenes) en la celda inferior:
1. Obtén el **promedio** y la **desviación estándar** de cada estudiante (`axis=1`).
2. Obtén la **mediana** y el **rango (`np.ptp`)** de cada examen (`axis=0`).
3. Calcula los percentiles **10** y **90** de toda la matriz de calificaciones.

In [13]:
notas_curso = np.array([
    [7.5, 8.0, 6.5, 9.0],
    [9.0, 9.5, 8.8, 9.2],
    [5.0, 6.0, 5.5, 7.0],
    [8.0, 7.5, 8.5, 8.0],
    [6.5, 7.0, 6.0, 7.5]
])

# Escribe tu solución aquí:

## 5.3 Escalamiento de Vectores y Matrices (Normalización sobre 100)

En análisis de datos y aprendizaje automático, las variables suelen medirse en unidades incompatibles (por ejemplo, una columna sobre 20 puntos, otra sobre 50 puntos y otra sobre 1000 puntos). Para compararlas o combinarlas, debemos **reescalar** los datos a un rango común, como **[0, 100]**.

### Dos Métodos Principales de Escalamiento sobre 100

1. **Escalamiento Proporcional al Máximo (Regla de Tres):**
   Conserva el cero absoluto y lleva el valor máximo de referencia exactamente a `100`:
   $$X_{\text{100}} = \left( \frac{X}{X_{\max}} \right) \times 100$$

2. **Escalamiento Min-Max (Normalización de Rango Completo):**
   Transforma el valor mínimo exactamente a `0` y el valor máximo exactamente a `100`:
   $$X_{\text{minmax}} = \left( \frac{X - X_{\min}}{X_{\max} - X_{\min}} \right) \times 100$$

### Sintaxis Mínima

In [20]:
# 1. Escalamiento de un Vector 1D: Puntajes obtenidos en un examen cuya nota máxima posible fue 45
puntajes_crudos = np.array([22.5, 36.0, 45.0, 18.0, 31.5, 40.5, 27.0, 38.25, 13.5, 42.75])

# Escala sobre 100 respecto al puntaje máximo de la prueba (45)
notas_sobre_100 = (puntajes_crudos / 45.0) * 100
print("Puntajes crudos (sobre 45):", puntajes_crudos)
print("Notas escaladas (sobre 100):", np.round(notas_sobre_100, 1))

# 2. Escalamiento de una Matriz 2D (4x3) donde cada COLUMNA tiene una escala máxima distinta
# Col 0: Asistencia (sobre 20) | Col 1: Talleres (sobre 50) | Col 2: Proyecto (sobre 200)
evaluaciones = np.array([
    [18, 40, 160],
    [15, 35, 140],
    [20, 50, 200],
    [12, 25, 110]
])
print("\nMatriz evaluaciones:\n", evaluaciones)
# Escalamiento proporcional por COLUMNA (axis=0): max_col tiene shape (3,), compatible con (4, 3)
max_por_columna = np.max(evaluaciones, axis=0)
print("\nMáximo por columna:\n", max_por_columna)
eval_sobre_100_col = (evaluaciones / max_por_columna) * 100
print("\nMatriz reescalada sobre 100 por Méximo de cada COLUMNA (axis=0):\n", np.round(eval_sobre_100_col, 1))

# 3. Escalamiento Min-Max por FILA (axis=1) usando keepdims=True
# Importante: keepdims=True mantiene shape (4, 1) para que el Broadcasting funcione por filas
min_fila = np.min(evaluaciones, axis=1, keepdims=True)
max_fila = np.max(evaluaciones, axis=1, keepdims=True)
print("\nMínimo por fila:\n", min_fila)
print("\nMáximo por fila:\n", max_fila)

minmax_por_fila = ((evaluaciones - min_fila) / (max_fila - min_fila)) * 100
print("\nMatriz normalizada Min-Max [0, 100] por FILA (axis=1):\n", np.round(minmax_por_fila, 1))

Puntajes crudos (sobre 45): [22.5  36.   45.   18.   31.5  40.5  27.   38.25 13.5  42.75]
Notas escaladas (sobre 100): [ 50.  80. 100.  40.  70.  90.  60.  85.  30.  95.]

Matriz evaluaciones:
 [[ 18  40 160]
 [ 15  35 140]
 [ 20  50 200]
 [ 12  25 110]]

Máximo por columna:
 [ 20  50 200]

Matriz reescalada sobre 100 por Méximo de cada COLUMNA (axis=0):
 [[ 90.  80.  80.]
 [ 75.  70.  70.]
 [100. 100. 100.]
 [ 60.  50.  55.]]

Mínimo por fila:
 [[18]
 [15]
 [20]
 [12]]

Máximo por fila:
 [[160]
 [140]
 [200]
 [110]]

Matriz normalizada Min-Max [0, 100] por FILA (axis=1):
 [[  0.   15.5 100. ]
 [  0.   16.  100. ]
 [  0.   16.7 100. ]
 [  0.   13.3 100. ]]


:::{warning} Trampa Común: Olvidar `keepdims=True` al escalar por filas (`axis=1`)
Si tienes una matriz `M` de forma `(4, 3)` y calculas `max_fila = np.max(M, axis=1)`, NumPy devuelve un vector 1D de forma `(4,)`. Al intentar dividir `M / max_fila`, NumPy lanzará `ValueError: operands could not be broadcast together with shapes (4,3) (4,)`.
**Regla de oro:** Siempre que vayas a usar el resultado de una reducción (`min`, `max`, `mean`, `std`) para operar de vuelta contra la matriz original, pasa el parámetro **`keepdims=True`** (ej. `np.max(M, axis=1, keepdims=True)`), lo que preserva la forma `(4, 1)` y garantiza compatibilidad automática con *Broadcasting*.
:::

:::{note} Restricción de División por Cero en Min-Max
Si toda una fila o columna tiene exactamente el mismo valor constante, entonces $X_{\max} - X_{\min} = 0$, lo que producirá `nan` al dividir. Asegúrate de que exista variación en el eje antes de aplicar Min-Max.
:::

### 🧩 Micro-reto 5.3
Tienes una matriz `metricas_atletas` de $4 \times 3$ (4 atletas $\times$ 3 pruebas físicas con escalas distintas: metros, segundos invertidos y puntos) definida en la celda inferior:
1. Aplica un **escalamiento Min-Max sobre 100 por columna (`axis=0`)** para llevar las tres pruebas a una escala comparable de `0` a `100` (utiliza `keepdims=True`).
2. Calcula el **puntaje promedio final sobre 100** de cada atleta (`np.mean(..., axis=1)` sobre la matriz ya escalada).

In [21]:
metricas_atletas = np.array([
    [6.8,  120, 45],
    [7.5,  150, 60],
    [6.2,  100, 30],
    [7.1,  135, 50]
])

# Escribe tu solución aquí:

## ✅ Resumen del Módulo 5
* En matrices 2D, **`axis=0`** colapsa las filas (entrega un resultado por columna) y **`axis=1`** colapsa las columnas (entrega un resultado por fila).
* Las funciones **`cumsum`** y **`cumprod`** preservan el `shape` original mostrando la evolución acumulada paso a paso.
* Para estadísticos muestrales insesgados, añade siempre **`ddof=1`** en `np.std()` y `np.var()`.
* Al normalizar o reescalar matrices (por ejemplo, llevando valores a escala **0–100**), incluye **`keepdims=True`** en `np.min()`, `np.max()` o `np.mean()` para evitar errores de dimensionalidad en el *Broadcasting*.